# 🧬 PMOS Intelligence Platform
## Block 1 — Exploratory Data Analysis (EDA)
**Goal:** Understand the dataset deeply before touching any model.

We will cover:
- Loading and inspecting the data
- Missing value analysis
- Target variable distribution
- Hormone and metabolic feature distributions
- Correlation analysis
- Outlier detection

---
### Cell 1 — Import Libraries

In [1]:
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import missingno as msno

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


---
### Cell 2 — Load Dataset

In [4]:
# Load the dataset
# Make sure PCOS_data.csv is inside data/raw/
df = pd.read_excel('../data/raw/PCOS_data_without_infertility.xlsx')

print(f'✅ Dataset loaded successfully')
print(f'   Shape: {df.shape[0]} rows × {df.shape[1]} columns')

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/PCOS_data_without_infertility.xlsx'

---
### Cell 3 — First Look at the Data

In [ ]:
# First 5 rows
print('=== FIRST 5 ROWS ===')
df.head()

In [ ]:
# Last 5 rows
print('=== LAST 5 ROWS ===')
df.tail()

In [ ]:
# Column names — see all features
print('=== ALL COLUMN NAMES ===')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

---
### Cell 4 — Data Types and Basic Info

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes)
print(f'\nNumerical columns : {df.select_dtypes(include=np.number).shape[1]}')
print(f'Categorical columns: {df.select_dtypes(include="object").shape[1]}')

In [ ]:
print('=== DATASET INFO ===')
df.info()

---
### Cell 5 — Duplicate Check

In [ ]:
duplicates = df.duplicated().sum()
print(f'=== DUPLICATE ROWS ===')
print(f'  Total duplicates: {duplicates}')

if duplicates > 0:
    df = df.drop_duplicates()
    print(f'  ✅ Duplicates removed. New shape: {df.shape}')
else:
    print(f'  ✅ No duplicates found')

---
### Cell 6 — Target Variable Analysis
**Target:** `PCOS (Y/N)` — this is what we are predicting

In [ ]:
# Identify target column
# The target column may be named differently — check and update if needed
target_col = 'PCOS (Y/N)'

print('=== TARGET VARIABLE DISTRIBUTION ===')
counts = df[target_col].value_counts()
percentages = df[target_col].value_counts(normalize=True) * 100

print(f'  PMOS Negative (0): {counts[0]} patients ({percentages[0]:.1f}%)')
print(f'  PMOS Positive (1): {counts[1]} patients ({percentages[1]:.1f}%)')
print(f'  Imbalance ratio  : {counts[0]/counts[1]:.2f}:1')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['PMOS Negative', 'PMOS Positive'], counts,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8)
axes[0].set_title('PMOS Diagnosis Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=['PMOS Negative', 'PMOS Positive'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, explode=(0, 0.05))
axes[1].set_title('PMOS Diagnosis Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable — PMOS Diagnosis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/plots/01_target_distribution.png', bbox_inches='tight')
plt.show()
print('\n⚠️  Class imbalance noted — will handle in Block 2 with SMOTE+ADASYN')

---
### Cell 7 — Missing Value Analysis

In [ ]:
print('=== MISSING VALUE ANALYSIS ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

# Show only columns with missing values
missing_df_filtered = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df_filtered) == 0:
    print('  ✅ No missing values found in dataset')
else:
    print(f'  ⚠️  {len(missing_df_filtered)} columns have missing values:\n')
    print(missing_df_filtered.to_string())

In [ ]:
# Missing value heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Missingno matrix
msno.matrix(df, ax=axes[0], sparkline=False, color=(0.8, 0.3, 0.3))
axes[0].set_title('Missing Value Matrix\n(white lines = missing)', 
                   fontsize=12, fontweight='bold')

# Missing % bar chart
if len(missing_df_filtered) > 0:
    missing_df_filtered['Missing %'].plot(kind='barh', ax=axes[1], 
                                           color='#e74c3c', edgecolor='black')
    axes[1].set_title('Missing Value % by Column', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Missing %')
else:
    axes[1].text(0.5, 0.5, '✅ No Missing Values!',
                 ha='center', va='center', fontsize=14,
                 color='green', fontweight='bold',
                 transform=axes[1].transAxes)
    axes[1].set_title('Missing Value % by Column', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/plots/02_missing_values.png', bbox_inches='tight')
plt.show()

---
### Cell 8 — Statistical Summary

In [ ]:
print('=== STATISTICAL SUMMARY (Numerical Features) ===')
df.describe().T

---
### Cell 9 — Hormone Feature Analysis
Key hormones in PMOS: LH, FSH, AMH, Testosterone, TSH, Prolactin

In [ ]:
# Define hormone columns — update names if different in your dataset
hormone_cols = [
    col for col in df.columns 
    if any(h in col.upper() for h in ['LH', 'FSH', 'AMH', 'TSH', 'PRL', 'TESTOSTERONE', 'PROLACTIN'])
]

print('=== HORMONE COLUMNS IDENTIFIED ===')
for col in hormone_cols:
    print(f'  → {col}')

if len(hormone_cols) == 0:
    print('  ⚠️  No hormone columns auto-detected.')
    print('  Please check column names above and update hormone_cols manually.')

In [ ]:
# Hormone distributions — PMOS positive vs negative
if len(hormone_cols) > 0:
    n_cols = 3
    n_rows = (len(hormone_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
    axes = axes.flatten()
    
    for i, col in enumerate(hormone_cols):
        # Plot distribution for PMOS+ and PMOS-
        for label, color, name in zip([0, 1], 
                                       ['#2ecc71', '#e74c3c'], 
                                       ['PMOS Negative', 'PMOS Positive']):
            subset = df[df[target_col] == label][col].dropna()
            axes[i].hist(subset, bins=25, alpha=0.6, color=color, 
                        label=name, edgecolor='white', linewidth=0.5)
        
        axes[i].set_title(col, fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Count')
        axes[i].legend(fontsize=8)
    
    # Hide empty subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Hormone Distributions — PMOS Positive vs Negative', 
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../outputs/plots/03_hormone_distributions.png', bbox_inches='tight')
    plt.show()

---
### Cell 10 — LH/FSH Ratio Analysis
The LH/FSH ratio is one of the most important PMOS diagnostic markers.
A ratio > 2 strongly suggests PMOS.

In [ ]:
# LH/FSH ratio — update column names if needed
lh_col  = [col for col in df.columns if 'LH' in col.upper() and 'FSH' not in col.upper()]
fsh_col = [col for col in df.columns if 'FSH' in col.upper() and 'LH'  not in col.upper()]

print(f'LH column  : {lh_col}')
print(f'FSH column : {fsh_col}')

if lh_col and fsh_col:
    lh_col  = lh_col[0]
    fsh_col = fsh_col[0]
    
    # Calculate ratio
    df['LH_FSH_Ratio'] = df[lh_col] / df[fsh_col]
    
    print(f'\n=== LH/FSH RATIO STATISTICS BY GROUP ===')
    print(df.groupby(target_col)['LH_FSH_Ratio'].describe().round(3))
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Boxplot
    df.boxplot(column='LH_FSH_Ratio', by=target_col, ax=axes[0],
               boxprops=dict(color='navy'),
               medianprops=dict(color='red', linewidth=2))
    axes[0].set_title('LH/FSH Ratio by PMOS Status', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('PMOS (0=Negative, 1=Positive)')
    axes[0].set_ylabel('LH/FSH Ratio')
    axes[0].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='Threshold = 2')
    axes[0].legend()
    plt.suptitle('')  # Remove auto title
    
    # Distribution
    for label, color, name in zip([0, 1], ['#2ecc71', '#e74c3c'], 
                                   ['PMOS Negative', 'PMOS Positive']):
        subset = df[df[target_col] == label]['LH_FSH_Ratio'].dropna()
        axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=name)
    axes[1].axvline(x=2, color='black', linestyle='--', linewidth=2, label='Clinical threshold = 2')
    axes[1].set_title('LH/FSH Ratio Distribution', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('LH/FSH Ratio')
    axes[1].set_ylabel('Count')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('../outputs/plots/04_LH_FSH_ratio.png', bbox_inches='tight')
    plt.show()
else:
    print('\n⚠️  LH or FSH column not found. Check column names and update manually.')

---
### Cell 11 — Metabolic Feature Analysis
BMI, Blood Pressure, Glucose, Insulin

In [ ]:
# Identify metabolic columns
metabolic_keywords = ['BMI', 'BP', 'BLOOD', 'GLUCOSE', 'INSULIN', 
                       'WEIGHT', 'WAIST', 'HIP', 'PULSE']
metabolic_cols = [
    col for col in df.columns 
    if any(k in col.upper() for k in metabolic_keywords)
]

print('=== METABOLIC COLUMNS IDENTIFIED ===')
for col in metabolic_cols:
    print(f'  → {col}')

# Boxplots — metabolic features by PMOS status
if len(metabolic_cols) > 0:
    n_cols = 3
    n_rows = (len(metabolic_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
    axes = axes.flatten()
    
    for i, col in enumerate(metabolic_cols):
        data_groups = [df[df[target_col] == 0][col].dropna(),
                       df[df[target_col] == 1][col].dropna()]
        bp = axes[i].boxplot(data_groups, labels=['PMOS-', 'PMOS+'],
                              patch_artist=True)
        bp['boxes'][0].set_facecolor('#2ecc7166')
        bp['boxes'][1].set_facecolor('#e74c3c66')
        for median in bp['medians']:
            median.set_color('black')
            median.set_linewidth(2)
        axes[i].set_title(col, fontsize=10, fontweight='bold')
        axes[i].set_ylabel('Value')
    
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Metabolic Features — PMOS Positive vs Negative',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('../outputs/plots/05_metabolic_features.png', bbox_inches='tight')
    plt.show()

---
### Cell 12 — Correlation Heatmap

In [ ]:
# Select numerical columns only
numerical_df = df.select_dtypes(include=np.number)

# Correlation matrix
corr_matrix = numerical_df.corr()

# Plot heatmap
plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show only lower triangle
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={'size': 7},
    vmin=-1, vmax=1
)
plt.title('Correlation Heatmap — All Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../outputs/plots/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top features correlated with PMOS target
print('=== TOP FEATURES CORRELATED WITH PMOS ===')
target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=abs, ascending=False)
print('\nTop 15 most correlated features:')
print(target_corr.head(15).to_string())

# Bar chart
plt.figure(figsize=(12, 6))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in target_corr.head(15)]
target_corr.head(15).plot(kind='barh', color=colors, edgecolor='black', linewidth=0.5)
plt.axvline(x=0, color='black', linewidth=1)
plt.title('Feature Correlation with PMOS Diagnosis', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.savefig('../outputs/plots/07_target_correlation.png', bbox_inches='tight')
plt.show()

---
### Cell 13 — Outlier Detection

In [ ]:
# IQR-based outlier detection
print('=== OUTLIER DETECTION (IQR Method) ===')
outlier_summary = []

for col in numerical_df.columns:
    if col == target_col:
        continue
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = (n_outliers / len(df)) * 100
    outlier_summary.append({
        'Feature': col,
        'Outliers': n_outliers,
        'Outlier %': round(pct, 2),
        'Lower Bound': round(lower, 3),
        'Upper Bound': round(upper, 3)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False)
print('\nTop 15 features with most outliers:')
print(outlier_df.head(15).to_string(index=False))

---
### Cell 14 — Symptom Analysis
Binary symptoms: hair loss, acne, weight gain, cycle irregularity

In [ ]:
# Identify symptom columns (binary 0/1 features)
binary_cols = [
    col for col in df.columns
    if df[col].dropna().isin([0, 1]).all() and col != target_col
]

print('=== BINARY SYMPTOM COLUMNS ===')
for col in binary_cols:
    print(f'  → {col}')

if len(binary_cols) > 0:
    # Symptom prevalence in PMOS+ vs PMOS-
    symptom_data = []
    for col in binary_cols:
        pmos_pos = df[df[target_col] == 1][col].mean() * 100
        pmos_neg = df[df[target_col] == 0][col].mean() * 100
        symptom_data.append({
            'Symptom': col,
            'PMOS Positive %': round(pmos_pos, 1),
            'PMOS Negative %': round(pmos_neg, 1),
            'Difference': round(pmos_pos - pmos_neg, 1)
        })
    
    symptom_df = pd.DataFrame(symptom_data).sort_values('Difference', ascending=False)
    print('\n=== SYMPTOM PREVALENCE BY PMOS STATUS ===')
    print(symptom_df.to_string(index=False))
    
    # Plot top symptoms
    top_symptoms = symptom_df.head(8)
    x = range(len(top_symptoms))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(14, 6))
    bars1 = ax.bar([i - width/2 for i in x], top_symptoms['PMOS Negative %'],
                   width, label='PMOS Negative', color='#2ecc71', edgecolor='black', linewidth=0.5)
    bars2 = ax.bar([i + width/2 for i in x], top_symptoms['PMOS Positive %'],
                   width, label='PMOS Positive', color='#e74c3c', edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Symptom')
    ax.set_ylabel('Prevalence (%)')
    ax.set_title('Symptom Prevalence — PMOS Positive vs Negative', 
                 fontsize=13, fontweight='bold')
    ax.set_xticks(list(x))
    ax.set_xticklabels(top_symptoms['Symptom'], rotation=30, ha='right', fontsize=9)
    ax.legend()
    ax.grid(axis='y', alpha=0.4)
    
    plt.tight_layout()
    plt.savefig('../outputs/plots/08_symptom_analysis.png', bbox_inches='tight')
    plt.show()

---
### Cell 15 — Save Cleaned Data & EDA Summary

In [ ]:
# Save cleaned dataframe for Block 2
df.to_csv('../data/processed/pmos_eda_clean.csv', index=False)
print('✅ Cleaned data saved to data/processed/pmos_eda_clean.csv')

# Final EDA Summary
print('\n' + '='*55)
print('         BLOCK 1 — EDA SUMMARY COMPLETE')
print('='*55)
print(f'  Total patients          : {len(df)}')
print(f'  Total features          : {df.shape[1]}')
print(f'  PMOS Positive patients  : {(df[target_col]==1).sum()}')
print(f'  PMOS Negative patients  : {(df[target_col]==0).sum()}')
print(f'  Class imbalance ratio   : {(df[target_col]==0).sum()/(df[target_col]==1).sum():.2f}:1')
print(f'  Missing values          : {df.isnull().sum().sum()}')
print(f'  Duplicate rows          : 0 (removed)')
print(f'  LH/FSH ratio created    : Yes')
print(f'  Plots saved to          : outputs/plots/')
print('='*55)
print('\n➡️   Next: Block 2 — Feature Selection & Preprocessing')

---
## ✅ Block 1 Complete!

### What we did:
- ✅ Loaded and inspected the PMOS dataset
- ✅ Checked for duplicates and missing values
- ✅ Analysed target variable distribution and class imbalance
- ✅ Explored hormone distributions (LH, FSH, AMH etc.)
- ✅ Computed LH/FSH ratio — key clinical marker
- ✅ Analysed metabolic features by PMOS status
- ✅ Built correlation heatmap
- ✅ Detected outliers using IQR method
- ✅ Analysed symptom prevalence
- ✅ Saved cleaned data for Block 2

### Key findings to note:
- Note your class imbalance ratio here
- Note which hormones showed biggest difference between groups
- Note how many missing values were found

### Next → Block 2: Feature Selection & Preprocessing